In [ ]:
%pip install -q openai pandas python-dotenv

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import importlib
import os
import pandas as pd

import openai_call_metrics
importlib.reload(openai_call_metrics)
from openai_call_metrics import (
    OpenAIMetricsSession,
    chat_completions_create_with_metrics,
    format_openai_session_batch_totals,
)

In [ ]:
# Loads OPENAI_API_KEY from a .env file in the project folder (if you use one).
load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "No OPENAI_API_KEY found. Do one of: "
        "(1) Create a .env file next to this notebook with: OPENAI_API_KEY=sk-... "
        "(2) Or export it before starting Jupyter: export OPENAI_API_KEY=sk-... "
        "Cursor/VSCode notebooks often do not inherit your terminal exports."
    )

In [ ]:
client = OpenAI(api_key=api_key)
openai_metrics = OpenAIMetricsSession()

In [ ]:
N_ROWS = 10  # load at least this many rows if you set `n` below higher
data = pd.read_csv("real_estate_ads 3.csv", nrows=N_ROWS)

## Parameters of the model
These knobs apply to many Chat Completions models. (Your exact request must follow what the API allows for the model you choose; see the note for `gpt-5-nano` at the end.)

### temperature
Purpose: Controls the randomness of the output.  
Range: 0 to 1.  
Effect: A lower value (closer to 0) makes the model more deterministic and more predictable. A higher value (closer to 1) increases randomness and variation.  
In context: A value like `0.7` is often described as a balance between creativity and control—more varied than 0, less wild than 1.

### Limiting response length: `max_tokens` vs `max_completion_tokens`
Purpose: Puts a cap on how long the model’s reply can be.  
Unit: **Tokens** (roughly word pieces, not the same as words).  
Effect: The model stops generating once it hits the limit, which can truncate the answer.  
- Many older or general models use the parameter name **`max_tokens`**.  
- Newer OpenAI models (including **`gpt-5-nano`**) use **`max_completion_tokens`** instead; sending `max_tokens` can return a 400 error.  
In this notebook we set **`max_completion_tokens`** to cap each reply. For **GPT-5 / reasoning**, **hidden reasoning** counts toward that cap. If you see **empty text** but **large `completion_tokens`** in the metrics, the cap is almost always **too low** — increase **`MAX_COMPLETION_TOKENS`** in the code cell (typical fix: **4k–8k+** for short structured outputs on this family). Higher values increase **cost** and **latency**.

### top_p
Purpose: Controls diversity via **nucleus sampling**.  
Range: 0 to 1.  
Effect: The model only considers the smallest set of next-token candidates whose combined probability mass reaches `top_p`. Lower values narrow the search; at `1`, no extra nucleus filtering is applied.  
In context: Often used together with `temperature` to tune how “wide” the sampling is.

In summary, these parameters together shape how the model answers: `temperature` balances creativity and control, the max-length parameter caps how long the reply can be (`max_tokens` on many models, or `max_completion_tokens` for newer ones), and `top_p` influences how wide the set of next-token options is. You adjust them to match the task; what the API actually allows **depends on the model** (see below).

### Note for the model used in *this* notebook (`gpt-5-nano`)
The current API for this model:
- **Does not** accept `max_tokens` — use **`max_completion_tokens`**.  
- **Does not** support custom **`temperature` or `top_p`** other than the model’s default; include them in the request only if a future API allows it, otherwise the call fails with 400.  

So the code cell below only passes **`max_completion_tokens`** (plus the required `model` and `messages`).


In [ ]:
# gpt-5-nano bills hidden reasoning inside completion tokens. If this is too low, you get
# empty message.content while usage shows high completion_tokens — raise this first.
MAX_COMPLETION_TOKENS = 8_192


def submit_request(content):
    return chat_completions_create_with_metrics(
        client,
        session=openai_metrics,
        model="gpt-5-nano",
        messages=[
            {
                "role": "system",
                "content": (
                    "You will be provided with unstructured descriptions of apartments for sale "
                    "from the secondary market. Extract features useful for price prediction and "
                    "return ONLY valid CSV text with two columns: feature,value."
                ),
            },
            {
                "role": "user",
                "content": content,
            },
        ],
        max_completion_tokens=MAX_COMPLETION_TOKENS,
    )

In [ ]:
# Re-run the client + openai_metrics cell first if you want a fresh cost/timing total.
n = 3  # first n rows of `data.description` (increase N_ROWS in the data cell if n > N_ROWS)
responses = []
for i in range(min(n, len(data))):
    responses.append(submit_request(data.description.iloc[i]))


In [ ]:
for i, response in enumerate(responses):
    print(f"========== ad index {i} ==========")
    print(response.choices[0].message.content)
    print()

print(format_openai_session_batch_totals(openai_metrics))
print()
print(openai_metrics.format_summary())